In [1]:
# @title Mise en place de l'environnement
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!git clone --branch solution https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv pip install --system -e ".[fastapi]"

env: MLFLOW_TRACKING_URI=https://dagshub.com/m09/landscape-classifier.mlflow
env: MPLBACKEND=notebook
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.8/17.8 MB 41.5 MB/s eta 0:00:00
Cloning into 'landscape-classifier'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (49/49), done.
Receiving objects: 100% (71/71), 4.72 MiB | 20.23 MiB/s, done.
remote: Total 71 (delta 29), reused 57 (delta 15), pack-reused 0 (from 0)
Resolving deltas: 100% (29/29), done.
/content/landscape-classifier
Using Python 3.13.15 environment at: /usr
Resolved 201 packages in 1.07s
   Building landscape-classifier @ file:///content/landscape-classifier         ⠙ Preparing packages... (0/73)                                                  warning: Found license classifier `License :: OSI Approved :: Apache Software License`. License classifiers are ambiguous and deprecated per PEP 639; projects should use `project.license` and `project.license

# MLflow, registre de modèles & déploiement

Cette démonstration déroule la solution des travaux pratiques du même nom, sur la branche `solution` du [dépôt `landscape-classifier`](https://github.com/shuuchuu/landscape-classifier/tree/solution)&nbsp;: publication d'un modèle sur le registre de modèles MLflow, utilisation de ce registre, puis service du modèle par une API FastAPI empaquetée avec Docker.

## Identifiants pour le serveur MLflow

Entrez vos identifiants DagsHub (le mot de passe est le jeton secret, pas votre mot de passe DagsHub)&nbsp;:

In [2]:
username = ""
token = ""

In [3]:
# @title À exécuter
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Récupération des données

In [4]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

remote: Enumerating objects: 2, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 2 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (2/2), 1.94 KiB | 1.94 MiB/s, done.
remote: Enumerating objects: 24289, done.
remote: Counting objects: 100% (24289/24289), done.
remote: Compressing objects: 100% (24289/24289), done.
remote: Total 24289 (delta 0), reused 24289 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (24289/24289), 341.65 MiB | 20.46 MiB/s, done.
Updating files: 100% (24336/24336), done.
✓ landscape — 24336 fichiers, 401M


## Publication d'un modèle sur un registre de modèles

Le script d'entraînement journalise les métriques avec [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) (`autolog`), ainsi que le jeu de données d'entraînement. Il enveloppe ensuite le modèle Keras dans un modèle `pyfunc` personnalisé (`WrappedModel`) qui embarque le prétraitement des images, le publie sur le registre de modèles sous le nom `dev.ml.landscape-classifier`, et donne l'alias `champion` à la nouvelle version.

In [5]:
!cat src/landscape_classifier/train.py

import os
from io import BytesIO
from pathlib import Path
from subprocess import run
from typing import Any

import yaml
from keras import Model, Sequential, layers, optimizers
from mlflow import MlflowClient, log_input, set_experiment, start_run
from mlflow.data.numpy_dataset import from_numpy
from mlflow.keras import autolog
from mlflow.pyfunc import PythonModel, PythonModelContext, log_model
from numpy import array, vstack
from pydantic import BaseModel

from landscape_classifier.data import LABEL_NAMES, get_images, process_image


def get_lenet(image_size: tuple[int, int], learning_rate: float) -> Model:
    def conv(filters: int, padding: str) -> layers.Conv2D:
        return layers.Conv2D(
            filters=filters, kernel_size=5, padding=padding, activation="sigmoid"
        )

    def pooling() -> layers.MaxPooling2D:
        return layers.MaxPooling2D()

    def dense(units: int, activation: str = "sigmoid") -> layers.Dense:
        return layers.Dense(units, activation=acti

Ses paramètres sont gérés par DVC, dans `params.yaml`&nbsp;:

In [6]:
!cat params.yaml

train_dir: train-data
image_size: [100, 100]
learning_rate: 0.0001
experiment: Landscape Classification
name: model
model_name: dev.ml.landscape-classifier
model_alias: champion
epochs: 3


In [7]:
!landscape-classifier

2026-10-08 19:59:53.759093: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026/10/08 20:00:03 INFO mlflow.tracking.fluent: Experiment with name 'Landscape Classification' does not exist. Creating a new experiment.
Traitement des dossiers:   0% 0/6 [00:00<?, ?it/s]
Dossier buildings:   0% 0/2191 [00:00<?, ?it/s]
Dossier buildings:   5% 108/2191 [00:00<00:01, 1073.49it/s]
Dossier buildings:  10% 220/2191 [00:00<00:01, 1097.54it/s]
Dossier buildings:  15% 332/2191 [00:00<00:01, 1107.48it/s]
Dossier buildings:  20% 443/2191 [00:00<00:01, 1107.95it/s]
Dossier buildings:  25% 554/2191 [00:00<00:01, 1106.43it/s]
Dossier buildings:  30% 665/2191 [00:00<00:01, 1106.19it/s]
Dossier buildings:  35% 777/2191 [00:00<00:01, 1108.03it/s]
Dossier buildings:  41% 888/2

## Utilisation du registre de modèles MLflow

### Création d'un client MLflow

L'adresse de tracking est bien récupérée depuis la variable d'environnement définie en haut du notebook&nbsp;:

In [8]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

'https://dagshub.com/m09/landscape-classifier.mlflow'

### Recherche de modèles

Tous les modèles enregistrés, avec pour chacun sa version la plus récente&nbsp;:

In [9]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

dev.ml.landscape-classifier 1


### Récupération d'un modèle depuis le registre de modèles

Un modèle MLflow peut avoir plusieurs *flavors*, autant de manières de le charger. Le nôtre a été publié comme un modèle `pyfunc` personnalisé&nbsp;: sa seule flavor est `python_function`, qui donne tout le pipeline (prétraitement des images, prédiction, mise en forme du résultat). Une flavor Keras n'aurait donné que le réseau, sans le prétraitement.

In [10]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

2026/10/08 20:02:58 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required: google-auth==2.61.0)
 - grpcio (current: 1.83.1, required: grpcio==1.84.0)
 - h5py (

['python_function']
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step


ClassificationResult(predicted=['street', 'buildings'], probabilities=[{'buildings': 0.08523174375295639, 'forest': 0.35532912611961365, 'glacier': 0.046192318201065063, 'mountain': 0.029078448191285133, 'sea': 0.027984049171209335, 'street': 0.4561842978000641}, {'buildings': 0.43386971950531006, 'forest': 0.10677625238895416, 'glacier': 0.03800349310040474, 'mountain': 0.1849798709154129, 'sea': 0.11484260112047195, 'street': 0.12152810394763947}])

### Ajout d'alias et d'étiquettes

Les alias et les étiquettes (tags) permettent d'organiser les modèles dans le registre de modèles MLflow. Les alias ciblent forcément une version en particulier alors que les étiquettes peuvent aussi cibler toutes les versions d'un modèle.

Le script d'entraînement a déjà donné l'alias `champion` à la dernière version. On ajoute un tag `domain` à la valeur `cv` au modèle et un tag `size` à la valeur `small` à sa dernière version&nbsp;:

In [11]:
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")
client.get_model_version_by_alias(MODEL_NAME, "champion")

<ModelVersion: aliases=['champion'], creation_timestamp=1791489764477, current_stage='None', deployment_job_state=None, description='LeNet entraîné sur `train-data` pendant 3 époque(s), val_accuracy=0.5628.', last_updated_timestamp=1791489765911, metrics=None, model_id=None, name='dev.ml.landscape-classifier', params=None, run_id='4fe4996f54fc41f0a81d28b3eca71758', run_link='', source='models:/m-6df4eea5d2414abc910242f59635232e', status='READY', status_message=None, tags={'git.commit': 'c862d552aac75b500c3fb683c59d115b96b1dcbb', 'size': 'small'}, user_id='', version='1', workspace='default'>

On peut alors récupérer le modèle par son alias plutôt que par sa version exacte&nbsp;:

In [12]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

2026/10/08 20:03:11 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required: google-auth==2.61.0)
 - grpcio (current: 1.83.1, required: grpcio==1.84.0)
 - h5py (

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 652ms/step


ClassificationResult(predicted=['street'], probabilities=[{'buildings': 0.08523058146238327, 'forest': 0.3553881049156189, 'glacier': 0.046190883964300156, 'mountain': 0.02907376177608967, 'sea': 0.027965281158685684, 'street': 0.45615145564079285}])

### Changement de phase d'un modèle

Pour promouvoir un modèle dans la chaîne de mise en production, la méthode recommandée est de copier la version voulue vers un nouveau nom de modèle, ici celui de l'environnement de préproduction&nbsp;:

In [13]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

Successfully registered model 'staging.ml.landscape-classifier'.
Copied version '1' of model 'dev.ml.landscape-classifier' to version '1' of model 'staging.ml.landscape-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1791489794244, current_stage='None', deployment_job_state=None, description='LeNet entraîné sur `train-data` pendant 3 époque(s), val_accuracy=0.5628.', last_updated_timestamp=1791489794244, metrics=None, model_id=None, name='staging.ml.landscape-classifier', params=None, run_id='4fe4996f54fc41f0a81d28b3eca71758', run_link='', source='models:/124455_dev.ml.landscape-classifier/1', status='READY', status_message=None, tags={'git.commit': 'c862d552aac75b500c3fb683c59d115b96b1dcbb', 'size': 'small'}, user_id='', version='1', workspace='default'>

## Création d'une API avec FastAPI

Le prétraitement est la fonction `process_image`, utilisée aussi bien à l'entraînement qu'à l'inférence (dans `WrappedModel`)&nbsp;: c'est le point auquel il faut faire extrêmement attention, un prétraitement différent entre entraînement et inférence dégrade les prédictions sans la moindre erreur.

In [14]:
!cat src/landscape_classifier/data.py

import pathlib
import typing

import numpy
import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split

LABEL_NAMES = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABEL_NAMES)}


def process_image(
    file: typing.BinaryIO | str | pathlib.Path, image_size: tuple[int, int]
) -> numpy.ndarray:
    return numpy.array(Image.open(file).resize(image_size))[None, ...]


def get_images(
    dir_path: pathlib.Path, image_size: tuple[int, int]
) -> tuple[numpy.ndarray, numpy.ndarray, numpy.ndarray, numpy.ndarray]:
    images = []
    labels = []

    for subdir_path in tqdm.tqdm(
        list(dir_path.iterdir()), desc="Traitement des dossiers"
    ):
        dir_name = subdir_path.name

        label = LABEL_TO_INDEX.get(dir_name)

        for image_path in tqdm.tqdm(
            list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False
        ):
            images.append(process_image

L'API charge le modèle une seule fois, au démarrage, depuis le registre (par son alias), décrit son type de retour avec [Pydantic](https://docs.pydantic.dev/latest/) et accepte des images [envoyées en POST](https://fastapi.tiangolo.com/tutorial/request-files/)&nbsp;:

In [15]:
!cat src/landscape_classifier/api.py

from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Final, Literal, TypedDict, cast

from fastapi import FastAPI, HTTPException, UploadFile
from mlflow.exceptions import MlflowException
from mlflow.pyfunc import PyFuncModel, load_model
from pydantic import BaseModel

from landscape_classifier.data import LABEL_NAMES

MODEL_NAME: Final = "dev.ml.landscape-classifier"
MODEL_ALIAS: Final = "champion"

model: PyFuncModel | None = None


@asynccontextmanager
async def lifespan(_: FastAPI) -> AsyncIterator[None]:  # noqa: RUF029
    global model
    try:
        model = load_model(model_uri=f"models:/{MODEL_NAME}@{MODEL_ALIAS}")
    except MlflowException:
        model = None
    yield


app = FastAPI(lifespan=lifespan)


Probabilities = TypedDict("Probabilities", dict.fromkeys(LABEL_NAMES, float))  # type: ignore


class ClassificationResult(BaseModel):
    predicted: list[Literal[tuple(LABEL_NAMES)]]  # type: ignore
    probabilities: li

### Lancement de l'API

On lance le serveur en arrière-plan, puis on attend qu'il réponde (le chargement du modèle peut prendre un moment)&nbsp;:

In [16]:
!nohup uvicorn landscape_classifier.api:app --port 8000 > api.log 2>&1 &

In [17]:
import time

import requests

for _ in range(300):
  try:
    requests.get("http://127.0.0.1:8000/docs", timeout=1)
    break
  except requests.ConnectionError:
    time.sleep(1)
print(Path("api.log").read_text())

INFO:     Started server process [22675]
INFO:     Waiting for application startup.

2026/10/08 20:03:31 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - absl-py (current: 1.4.0, required: absl-py==2.5.0)
 - aiohttp (current: 3.14.3, required: aiohttp==3.14.4)
 - anyio (current: 4.14.2, required: anyio==4.15.1)
 - atpublic (current: 5.1, required: atpublic==8.0.1)
 - cachetools (current: 6.2.6, required: cachetools==7.2.1)
 - charset-normalizer (current: 3.4.9, required: charset-normalizer==3.5.2)
 - contourpy (current: 1.3.3, required: contourpy==1.4.0)
 - cryptography (current: 50.0.1, required: cryptography==50.0.2)
 - fastapi (current: 0.141.1, required: fastapi==0.143.0)
 - filelock (current: 3.32.5, required: filelock==4.0.12)
 - fonttools (current: 4.64.0, required: fonttools==4.66.1)
 - fsspec (current: 2025.12.0, required: fsspec==2026.9.0)
 - google-auth (current: 2.49.0, required:

### Appel de l'API

In [18]:
with open("forest1.jpg", "rb") as forest1, open("forest2.jpg", "rb") as forest2:
  response = requests.post(
    "http://127.0.0.1:8000/",
    files=[("images", forest1), ("images", forest2)],
  )
response.raise_for_status()
response.json()

{'predicted': ['street', 'buildings'],
 'probabilities': [{'buildings': 0.08523174375295639,
   'forest': 0.35532912611961365,
   'glacier': 0.046192318201065063,
   'mountain': 0.029078448191285133,
   'sea': 0.027984049171209335,
   'street': 0.4561842978000641},
  {'buildings': 0.43386971950531006,
   'forest': 0.10677625238895416,
   'glacier': 0.03800349310040474,
   'mountain': 0.1849798709154129,
   'sea': 0.11484260112047195,
   'street': 0.12152810394763947}]}

## Création d'un Dockerfile

Docker n'est pas disponible dans Colab&nbsp;: voici le Dockerfile de la solution. Une première étape exporte les dépendances avec `uv`, la seconde les installe sur l'image de base Python 3.12 avec le projet, puis lance le serveur FastAPI.

In [19]:
!cat Dockerfile

FROM python:3.12-slim-trixie AS base
WORKDIR /app

FROM base AS uv
COPY --from=ghcr.io/astral-sh/uv:0.12.14 /uv /uvx /bin/
COPY uv.lock pyproject.toml /app/
RUN uv export --extra fastapi --frozen --no-dev --no-emit-project -o requirements.txt

FROM base AS runtime
COPY --from=uv /app/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r requirements.txt
COPY ./pyproject.toml ./README.md /app/
COPY ./params.yaml /app/
COPY ./src /app/src/
RUN pip install .
CMD ["uvicorn", "landscape_classifier.api:app", "--host", "0.0.0.0", "--port", "80"]


Sur une machine avec Docker, on construit puis lance l'image en lui passant les identifiants MLflow en variables d'environnement&nbsp;:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

La documentation de l'API, qui sert aussi de client de test, est alors à l'adresse <http://localhost:8000/docs>.

## Amélioration de l'API

Plusieurs choix de la solution rendent l'API adaptée à la mise en production par image Docker&nbsp;:

- le modèle est chargé une seule fois, au démarrage, et non à chaque requête&nbsp;;
- les identifiants MLflow sont passés au lancement du conteneur, jamais copiés dans l'image&nbsp;;
- l'API répond 503 si le modèle n'a pas pu être chargé.

On pourrait aller plus loin en rendant le nom et l'alias du modèle configurables par variables d'environnement, pour servir la même image en développement et en préproduction.